In [100]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import collections
import random

In [101]:
#Hyperparameters
learning_rate = 0.0005
gamma         = 0.99
buffer_limit  = 50000
batch_size    = 32


In [102]:
class ReplayBuffer():
    def __init__(self):
        self.buffer = collections.deque(maxlen=buffer_limit)
    
    def put(self, transition):
        self.buffer.append(transition)
    
    def sample(self, n):
        mini_batch = random.sample(self.buffer, n)
        s_lst, a_lst, r_lst, s_prime_lst  = [], [], [], []
        
        for transition in mini_batch:
            s, a, r, s_prime = transition
            s_lst.append(s)
            a_lst.append([a])
            r_lst.append([r])
            s_prime_lst.append(s_prime)

        return torch.tensor(s_lst, dtype=torch.float), torch.tensor(a_lst), \
               torch.tensor(r_lst), torch.tensor(s_prime_lst, dtype=torch.float)    
    def size(self):
        return len(self.buffer)


In [103]:
class Qnet(nn.Module):
    def __init__(self, state_dim, action_dim, inner_dim):
        super(Qnet, self).__init__()
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.inner_dim = inner_dim
        self.fc1 = nn.Linear(state_dim, inner_dim)
        self.fc2 = nn.Linear(inner_dim, inner_dim)
        self.fc3 = nn.Linear(inner_dim, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
      
    def sample_action(self, obs, epsilon):
        out = self.forward(obs)
        coin = random.random()
        if coin < epsilon:
            return random.randint(0,self.action_dim - 1)
        else : 
            return out.argmax().item()
            

In [104]:
def train(q, q_target, memory, optimizer):
    for i in range(10):
        s,a,r,s_prime = memory.sample(batch_size)

        q_out = q(s)
        q_a = q_out.gather(1,a)
        max_q_prime = q_target(s_prime).max(1)[0].unsqueeze(1)
        target = r + gamma * max_q_prime 
        loss = F.smooth_l1_loss(q_a, target)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [105]:
class NW:
    def __init__(self, N, hs, ps):
        self.N = N # N is the total number of sources in the network - indexed from 0 to N - 1
        self.hs = hs # this is a list of hop distances of the sources
        self.ps = ps # modelling all the links to have a single ps
        
        self._initialize_ages()
        self._initialize_mapper()
        
    def _initialize_ages(self):
        self.ages = {}
        for i in range(self.N):
            age_source = [0] * self.hs[i]
            self.ages[i] = np.array(age_source)
                
        self.statespace_dimension = np.sum(self.hs)
        self.actionspace_dimension = np.sum(self.hs)

    def _initialize_mapper(self):
        self.maptosource = []
        self.maptolink = []
        for i in range(self.N):
            for j in range(self.hs[i]):
                self.maptosource.append(i)
                self.maptolink.append(j)

    def _flattenedages(self):
        ages = []
        for i in range(self.N):
            ages.extend(self.ages[i])

        return np.array(ages)
        
    def reset(self):
        self._initialize_ages()
        return self._flattenedages(), self.ages
    
    def step(self, action):
        scheduled_source = action[0]
        scheduled_link = action[1]
        
        for i in range(self.N):
            new_ages = self.ages[i] + 1
            if i == scheduled_source:
                if scheduled_link != (self.hs[i] - 1):
                    if np.random.rand() <= self.ps:
                        new_ages[scheduled_link] = self.ages[i][scheduled_link + 1] + 1
                else:
                    new_ages[scheduled_link] = 1
            self.ages[i] = new_ages
            
        return self._flattenedages(), self.ages


In [116]:
# env = NW(3, [1,2,3], 1)
env = NW(10, [1,2,3,4,5,6,7,8,9,10], 1)

q = Qnet(env.statespace_dimension, env.actionspace_dimension,  5 * env.statespace_dimension + 5 * env.actionspace_dimension)
q_target = Qnet(env.statespace_dimension, env.actionspace_dimension, 5 * env.statespace_dimension + 5 * env.actionspace_dimension)
q_target.load_state_dict(q.state_dict())
memory = ReplayBuffer()

print_interval = 100
score = 0.0  
optimizer = optim.Adam(q.parameters(), lr=learning_rate)

fs, s = env.reset()
num_runs = 10000

scores = []

for n in range(num_runs):
    epsilon = max(0.01, 0.1 - 0.01*(n/1000)) #Linear annealing from 8% to 1%
    a = q.sample_action(torch.from_numpy(fs).float(), epsilon)
    scheduled_source = env.maptosource[a]
    scheduled_link = env.maptolink[a]
    fs_prime, s_prime = env.step([scheduled_source, scheduled_link])
    r = 0
    for i in range(env.N):
        r = r - s[i][0]
        
    memory.put((fs, a, r/100.0, fs_prime))
    fs = fs_prime
    s = s_prime

    score += r
        
    if memory.size()>2000:
        train(q, q_target, memory, optimizer)

    if n % print_interval==0 and n !=0:
        q_target.load_state_dict(q.state_dict())
        print("n_episode :{}, score : {:.1f}, n_buffer : {}, eps : {:.1f}%".format(
                                                        n, score/print_interval, memory.size(), epsilon*100))
        scores.append(-score/print_interval)
        score = 0.0

n_episode :100, score : -515.1, n_buffer : 101, eps : 9.9%
n_episode :200, score : -1515.0, n_buffer : 201, eps : 9.8%
n_episode :300, score : -2515.0, n_buffer : 301, eps : 9.7%
n_episode :400, score : -3515.0, n_buffer : 401, eps : 9.6%
n_episode :500, score : -4515.0, n_buffer : 501, eps : 9.5%
n_episode :600, score : -5515.0, n_buffer : 601, eps : 9.4%
n_episode :700, score : -6515.0, n_buffer : 701, eps : 9.3%
n_episode :800, score : -7417.5, n_buffer : 801, eps : 9.2%
n_episode :900, score : -8002.0, n_buffer : 901, eps : 9.1%
n_episode :1000, score : -9002.0, n_buffer : 1001, eps : 9.0%
n_episode :1100, score : -10002.0, n_buffer : 1101, eps : 8.9%
n_episode :1200, score : -10943.5, n_buffer : 1201, eps : 8.8%
n_episode :1300, score : -11181.4, n_buffer : 1301, eps : 8.7%
n_episode :1400, score : -11685.0, n_buffer : 1401, eps : 8.6%
n_episode :1500, score : -11917.7, n_buffer : 1501, eps : 8.5%
n_episode :1600, score : -12122.9, n_buffer : 1601, eps : 8.4%
n_episode :1700, scor

KeyboardInterrupt: 

In [113]:
# Evaluate the policy
fs, s = env.reset()
num_runs = 1000
score = 0
for n in range(num_runs):
    a = q.sample_action(torch.from_numpy(fs).float(), 0)
    # print(n, s, a)
    scheduled_source = env.maptosource[a]
    scheduled_link = env.maptolink[a]
    fs_prime, s_prime = env.step([scheduled_source, scheduled_link])
    r = 0
    for i in range(env.N):
        r = r - s[i][0]
    
    score += r
    fs = fs_prime
    s = s_prime

print(-score/num_runs)

7008.006
